# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ladinski/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My Rule and Its Reason Codes

**Lane:** Refresh / Content Opportunity Scoring

**Objective:** Prioritize existing web pages for search-snippet review using observable search performance signals.

### Proposed Baseline Rule

A page qualifies for review when:

1. Its average search position is between 4 and 20, using the position buckets defined in the signal audit.
2. Its CTR is below the median CTR of pages within the same search-position bucket.
3. It has received at least 100 search impressions over the previous 90 days.

Eligible pages are ranked by total search impressions, prioritizing pages with higher search visibility.

### Baseline Score

- Eligible page: `baseline_score = impressions_90d`
- Ineligible page: `baseline_score = 0`

### Reason Codes

- `LOW_CTR_VISIBLE` — The page meets all three eligibility conditions.
- `NOT_FLAGGED` — The page does not meet the review criteria.

### Action Labels

- `Review search snippet` — Investigate whether the title, meta description, or search intent warrants attention.
- `No priority review` — The page is not prioritized by this particular baseline.

### Evidence Supporting the Rule

Two signals were investigated:

- **Content staleness — MIXED:** Decline rates were not consistently higher for older content.
- **CTR relative to position — MIXED:** Below-median CTR was associated with higher decline rates at positions 4–20, but not across all positions.

The selected rule identified 5,445 eligible pages (18.15% of the dataset), with an observed declining rate of 70.65%.

This is a transparent baseline rather than a trained ML model. Decline labels are used only for evaluation, not to calculate the score.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from pathlib import Path
import subprocess

repo_dir = Path("/content/flyrank-ml-internship")

if not (repo_dir / "data/raw/content_refresh_anonymized.csv").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Ladinski/flyrank-ml-internship.git",
        str(repo_dir)
    ], check=True)

df = pd.read_csv(
    repo_dir / "data/raw/content_refresh_anonymized.csv"
)

df["is_declining"] = (
    df["trend_direction"] == "down"
).astype(int)

print(f"Pages: {len(df):,}")
print(f"Overall declining rate: {df['is_declining'].mean():.1%}")
print("Dataset loaded successfully.")


Pages: 30,000
Overall declining rate: 54.2%
Dataset loaded successfully.


In [2]:

# Signal 1: Does content staleness relate to decline?

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days",
            "91-180 days", "180+ days"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=True)
    .agg(
        n=("is_declining", "size"),
        declining_pages=("is_declining", "sum"),
        declining_rate=("is_declining", "mean")
    )
)

staleness_table["declining_rate"] = (
    staleness_table["declining_rate"] * 100
).round(2)

display(staleness_table)


,n,declining_pages,declining_rate
staleness_bucket,,,
0-30 days,20480,10473,51.14
31-90 days,175,103,58.86
91-180 days,9171,5604,61.11
180+ days,174,82,47.13


In [3]:

# Signal 2: Low CTR relative to search position

# Divide pages into search position buckets
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

# Compare each page's CTR with the median
# CTR of pages in the same position bucket
df["bucket_median_ctr"] = (
    df.groupby("position_bucket", observed=True)["ctr"]
    .transform("median")
)

df["ctr_signal"] = np.where(
    df["ctr"] < df["bucket_median_ctr"],
    "Below median CTR",
    "At/above median CTR"
)

ctr_table = (
    df.groupby(["position_bucket", "ctr_signal"], observed=True)
    .agg(
        n=("is_declining", "size"),
        declining_rate=("is_declining", "mean")
    )
    .reset_index()
)

ctr_table["declining_rate"] = (
    ctr_table["declining_rate"] * 100
).round(2)

display(ctr_table)


,position_bucket,ctr_signal,n,declining_rate
0,1-3,At/above median CTR,2346,24.55
1,4-10,At/above median CTR,5955,53.65
2,4-10,Below median CTR,5887,60.27
3,11-20,At/above median CTR,3760,57.55
4,11-20,Below median CTR,3513,64.59
5,21-50,At/above median CTR,3664,59.91
6,21-50,Below median CTR,3561,52.34
7,50+,At/above median CTR,1314,34.32


### Signal Audit — Results

**Signal 1: Content staleness — MIXED**

The declining rate increased from 51.14% among pages updated within 30 days to 61.11% among pages last updated 91–180 days ago. However, pages older than 180 days had a lower declining rate of 47.13%, with only 174 observations.

Staleness does not show a consistently increasing relationship with decline.

**Signal 2: CTR relative to search position — MIXED**

Below-median CTR was associated with higher declining rates in positions 4–10 (+6.62 percentage points) and 11–20 (+7.04 percentage points).

However, the relationship reversed in positions 21–50 (−7.57 percentage points).

This suggests that a position-specific CTR signal may be more useful than a universal low-CTR threshold.

**Baseline decision:** I will focus on low CTR within positions 4–20, where the observed association is directionally consistent, and use search impressions to prioritize pages with greater visibility.

These observations are descriptive, not causal, and do not guarantee that improving CTR will reverse declining performance.

In [4]:

# Count pages eligible for our proposed baseline

eligible = (
    df["position_bucket"].isin(["4-10", "11-20"])
    & (df["ctr_signal"] == "Below median CTR")
    & (df["impressions_90d"] >= 100)
)

print("Eligible pages:", eligible.sum())
print("Eligible percentage:", f"{eligible.mean():.2%}")
print("Declining rate among eligible pages:",
      f"{df.loc[eligible, 'is_declining'].mean():.2%}")


Eligible pages: 5445
Eligible percentage: 18.15%
Declining rate among eligible pages: 70.65%


In [5]:

import numpy as np
from pathlib import Path

# Eligibility based only on observable signals
eligible = (
    df["position_bucket"].isin(["4-10", "11-20"])
    & (df["ctr_signal"] == "Below median CTR")
    & (df["impressions_90d"] >= 100)
)

# One transparent baseline score
df["baseline_score"] = np.where(
    eligible,
    df["impressions_90d"],
    0
)

# One reason code and action label
df["reason_code"] = np.where(
    eligible,
    "LOW_CTR_VISIBLE",
    "NOT_FLAGGED"
)

df["action_label"] = np.where(
    eligible,
    "Review search snippet",
    "No priority review"
)

# Rank pages, using original row order to resolve ties
queue = (
    df.sort_values(
        "baseline_score",
        ascending=False,
        kind="stable"
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

# Evaluate top 10 and top 50 (evaluation only)
for k in [10, 50]:
    top = queue.head(k)
    print(f"Precision@{k}: {top['is_declining'].mean():.3f}")

# Save to work/outputs (ignored by Git)
output_dir = repo_dir / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print(f"\nSaved {len(queue):,} rows to:")
print(output_path)

# Display top 10 without identifiers
display(queue[[
    "rank",
    "impressions_90d",
    "avg_position",
    "ctr",
    "baseline_score",
    "reason_code",
    "action_label"
]].head(10))


Precision@10: 0.400
Precision@50: 0.520

Saved 30,000 rows to:
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


,rank,impressions_90d,avg_position,ctr,baseline_score,reason_code,action_label
0,1,517715,4.2,0.14,517715,LOW_CTR_VISIBLE,Review search snippet
1,2,295097,7.3,0.05,295097,LOW_CTR_VISIBLE,Review search snippet
2,3,223271,7.8,0.03,223271,LOW_CTR_VISIBLE,Review search snippet
3,4,213963,4.7,0.10,213963,LOW_CTR_VISIBLE,Review search snippet
4,5,208678,9.7,0.00,208678,LOW_CTR_VISIBLE,Review search snippet
5,6,201111,5.7,0.11,201111,LOW_CTR_VISIBLE,Review search snippet
6,7,160851,7.8,0.12,160851,LOW_CTR_VISIBLE,Review search snippet
7,8,159590,7.8,0.06,159590,LOW_CTR_VISIBLE,Review search snippet
8,9,152968,9.1,0.15,152968,LOW_CTR_VISIBLE,Review search snippet
9,10,152467,6.5,0.13,152467,LOW_CTR_VISIBLE,Review search snippet


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:

# Generate a review table for the top 20 pages

top20 = queue.head(20).copy()

def review_risk(row):
    if row["ctr"] == 0:
        return "Zero CTR may reflect query intent or data quality."
    elif row["avg_position"] >= 10:
        return "Lower ranking position may explain reduced CTR."
    elif row["ctr"] < 0.05:
        return "Low CTR may result from search intent or SERP features."
    else:
        return "CTR may be reasonable for this specific query mix."

top20["review_action"] = "Review search snippet"

top20["confidence_note"] = (
    "Rule-based priority; no calibrated confidence estimate"
)

top20["why_selected"] = top20.apply(
    lambda row: (
        f"{row['impressions_90d']:,.0f} impressions; "
        f"position {row['avg_position']:.1f}; "
        f"CTR {row['ctr']:.2%}; below bucket median"
    ),
    axis=1
)

top20["what_could_make_it_wrong"] = top20.apply(
    review_risk, axis=1
)

review_cols = [
    "rank",
    "review_action",
    "reason_code",
    "why_selected",
    "confidence_note",
    "what_could_make_it_wrong"
]

display(top20[review_cols])


,rank,review_action,reason_code,why_selected,confidence_note,what_could_make_it_wrong
0,1,Review search snippet,LOW_CTR_VISIBLE,"517,715 impressions; position 4.2; CTR 14.00%;...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...
1,2,Review search snippet,LOW_CTR_VISIBLE,"295,097 impressions; position 7.3; CTR 5.00%; ...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...
2,3,Review search snippet,LOW_CTR_VISIBLE,"223,271 impressions; position 7.8; CTR 3.00%; ...",Rule-based priority; no calibrated confidence ...,Low CTR may result from search intent or SERP ...
3,4,Review search snippet,LOW_CTR_VISIBLE,"213,963 impressions; position 4.7; CTR 10.00%;...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...
4,5,Review search snippet,LOW_CTR_VISIBLE,"208,678 impressions; position 9.7; CTR 0.00%; ...",Rule-based priority; no calibrated confidence ...,Zero CTR may reflect query intent or data qual...
5,6,Review search snippet,LOW_CTR_VISIBLE,"201,111 impressions; position 5.7; CTR 11.00%;...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...
6,7,Review search snippet,LOW_CTR_VISIBLE,"160,851 impressions; position 7.8; CTR 12.00%;...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...
7,8,Review search snippet,LOW_CTR_VISIBLE,"159,590 impressions; position 7.8; CTR 6.00%; ...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...
8,9,Review search snippet,LOW_CTR_VISIBLE,"152,968 impressions; position 9.1; CTR 15.00%;...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...
9,10,Review search snippet,LOW_CTR_VISIBLE,"152,467 impressions; position 6.5; CTR 13.00%;...",Rule-based priority; no calibrated confidence ...,CTR may be reasonable for this specific query ...


## 4. Weak Picks and Leakage Check

### Baseline Performance

The baseline achieved:

- Precision@10: 0.400
- Precision@50: 0.520

Only 4 of the top 10 and 26 of the top 50 recommendations were labeled as declining.

### Weaknesses

**1. Impression-heavy ranking:** The baseline prioritizes search exposure without considering the severity of CTR underperformance.

**2. Position-bucket limitations:** Broad buckets may group pages with substantially different expected CTR values.

**3. Search intent:** Low CTR may reflect user intent or search-result features rather than a problem that requires editing.

**4. Missing context:** The rule does not incorporate detailed query-level behavior or content quality.

**5. Limited generalization evidence:** The current baseline was developed and evaluated using the same starter dataset. Performance on unseen clients and future periods has not been established.

### Leakage Check

The ranking score uses only:

- Search impressions.
- Average search position.
- CTR relative to its position bucket.

The target `is_declining`, `trend_direction`, and `trend_pct` are excluded from score construction.

These outcome-related fields are used only to evaluate the resulting ranking.

The CTR bucket medians were calculated from the full starter dataset. For future validation, any learned thresholds or reference statistics should be calculated using training data only.

### Conclusion

The baseline provides an interpretable starting point for editorial review prioritization.

Its performance indicates substantial room for improvement through better signal combinations, appropriate holdout evaluation, and more informative ranking models.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:

assert len(top20) == 20
assert len(queue) == 30000
assert queue["rank"].head(20).tolist() == list(range(1, 21))

print("Top-20 review and ranked queue verified!")


Top-20 review and ranked queue verified!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.